# Data Wrangling Part I: Cleaning

## Objective
This notebook builds a reproducible cleaning pipeline for the merged country-year dataset. The goal is to make the dataset consistent, trustworthy, and ready for later modeling and visualization.

The cleaning process focuses on:
- fixing data types
- standardizing text values
- checking duplicates
- documenting and handling missing values
- identifying obvious errors and invalid ranges
- saving a clean analysis-ready dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/merged_dataset.csv")

print("Initial shape:", df.shape)
df.head()

Initial shape: (7820, 24)


,country_code,country,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,...,sanction_active,sanction_type,sanction_duration,sanction_intensity_index,regime_score,conflict_incidence,conflict_intensity,oil_exports,pharma_imports,fuel_imports
0,ABW,ABW,1995,3.361391,2.547144,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,ABW,1996,3.225288,1.185789,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,ABW,1997,2.999948,7.046875,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,ABW,1998,1.869489,1.991984,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,ABW,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Initial Inspection

I begin by loading the merged dataset and checking its structure. Since this project uses a country-year panel, the first step is to confirm that the file loads correctly and that the variables appear in a usable format.

At this stage, I am not trying to fix everything immediately. The purpose is to understand the dataset before making cleaning decisions.

In [3]:
print("Dataset info:\n")
print(df.info())

print("\nColumns:\n")
print(df.columns.tolist())

print("\nDuplicate country-year rows:")
print(df[["country_code", "year"]].duplicated().sum())

print("\nMissing values by column:\n")
print(df.isna().sum().sort_values(ascending=False))

Dataset info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7820 entries, 0 to 7819
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   country_code              7820 non-null   object 
 1   country                   7820 non-null   object 
 2   year                      7820 non-null   int64  
 3   inflation_rate            6740 non-null   float64
 4   gdp_growth                7588 non-null   float64
 5   school_enrollment         3753 non-null   float64
 6   SE.PRM.NENR.FE            2880 non-null   float64
 7   SE.PRM.NENR.MA            2880 non-null   float64
 8   child_mortality_u5        7076 non-null   float64
 9   SH.DYN.MORT.FE            7076 non-null   float64
 10  SH.DYN.MORT.MA            7076 non-null   float64
 11  poverty_rate              2471 non-null   float64
 12  gini_index                2059 non-null   float64
 13  unemployment_rate         7041 non-null   float6

The initial inspection confirms that the dataset contains 7,820 country-year observations and 24 variables. It also shows that the merged structure is intact and that country_code and year can be used as the panel identifiers.

The missingness summary shows that some socioeconomic, political, and trade variables have substantial missing values. This is expected because the dataset was created by merging multiple sources with different country and year coverage. The next steps focus on making types and formats consistent before applying variable-specific cleaning rules.

## Cleaning Plan

The merged dataset is already structurally correct, but it still needs cleaning before it can be used for modeling. The cleaning pipeline below follows a reproducible sequence:

1. Create a working copy of the data
2. Convert columns to appropriate data types
3. Standardize text formatting
4. Check and remove duplicates if necessary
5. Quantify missing values and apply justified rules
6. Detect obvious errors and invalid ranges
7. Validate the cleaned output and save it

In [4]:
clean_df = df.copy()

## Data Type Fixes

Before cleaning values, I first make sure that each column has a sensible data type. This is important because numeric operations, comparisons, and summaries can behave incorrectly when columns are stored as text.

The `year` column should behave like an integer time variable. Economic, social, political, and trade indicators should be numeric. Text-based fields such as `country_code`, `country`, and `sanction_type` should be standardized as strings.

In [5]:
# year
clean_df["year"] = pd.to_numeric(clean_df["year"], errors="coerce").astype("Int64")

# numeric columns
numeric_cols = [
    "inflation_rate", "gdp_growth", "school_enrollment",
    "SE.PRM.NENR.FE", "SE.PRM.NENR.MA",
    "child_mortality_u5", "SH.DYN.MORT.FE", "SH.DYN.MORT.MA",
    "poverty_rate", "gini_index", "unemployment_rate",
    "sanction_active", "sanction_duration", "sanction_intensity_index",
    "regime_score", "conflict_incidence", "conflict_intensity",
    "oil_exports", "pharma_imports", "fuel_imports"
]

for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# string / categorical-style columns
clean_df["country_code"] = clean_df["country_code"].astype(str).str.strip().str.upper()
clean_df["country"] = clean_df["country"].astype(str).str.strip().str.upper()
clean_df["sanction_type"] = clean_df["sanction_type"].astype(str).str.strip().str.lower()

print(clean_df.dtypes)

country_code                 object
country                      object
year                          Int64
inflation_rate              float64
gdp_growth                  float64
school_enrollment           float64
SE.PRM.NENR.FE              float64
SE.PRM.NENR.MA              float64
child_mortality_u5          float64
SH.DYN.MORT.FE              float64
SH.DYN.MORT.MA              float64
poverty_rate                float64
gini_index                  float64
unemployment_rate           float64
sanction_active             float64
sanction_type                object
sanction_duration           float64
sanction_intensity_index    float64
regime_score                float64
conflict_incidence          float64
conflict_intensity          float64
oil_exports                 float64
pharma_imports              float64
fuel_imports                float64
dtype: object


The updated data types confirm that the panel identifiers and measurement variables are now stored in a more reliable format. This reduces the risk of hidden type issues affecting later cleaning, validation, or analysis.

## Standardization of Text Values

String columns often contain hidden inconsistencies such as casing differences, extra spaces, or inconsistent labels. Even when the dataset appears clean, it is good practice to standardize categorical values so that later grouping, filtering, and encoding steps behave consistently.

In [6]:
clean_df["sanction_type"] = clean_df["sanction_type"].replace({
    "nan": np.nan,
    "none": "none"
})

clean_df["country"] = clean_df["country"].replace({
    "NAN": np.nan
})

print("Unique sanction_type values:")
print(clean_df["sanction_type"].value_counts(dropna=False))

Unique sanction_type values:
sanction_type
none         7507
trade         130
financial      96
targeted       87
Name: count, dtype: int64


The sanction categories are now standardized into a small set of clean labels. This is useful because `sanction_type` will later be encoded as a categorical feature during feature engineering.

## Duplicate Handling

Because this dataset is a country-year panel, the correct definition of uniqueness is one row per `country_code` and `year`. I check for duplicate country-year rows and remove them only if they exist. Even if no duplicates are present, this remains an important validation step because it confirms that the panel structure is valid.

In [7]:
rows_before = clean_df.shape[0]

clean_df = clean_df.drop_duplicates(subset=["country_code", "year"], keep="first")

rows_after = clean_df.shape[0]

print("Rows before duplicate removal:", rows_before)
print("Rows after duplicate removal:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

Rows before duplicate removal: 7820
Rows after duplicate removal: 7820
Duplicates removed: 0


No duplicate country-year rows were found. This confirms that the merged dataset already preserves a clean panel structure, which is important for later modeling.


## Missing-Value Summary

Before deciding how to handle missing values, I first quantify how much missingness is present in each column. This helps identify which variables are relatively complete and which ones may require special treatment later.

In [8]:
missing_summary = clean_df.isna().sum().sort_values(ascending=False).reset_index()
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(clean_df) * 100).round(2)

missing_summary

,column,missing_count,missing_percent
0,gini_index,5761,73.67
1,poverty_rate,5349,68.40
2,SE.PRM.NENR.MA,4940,63.17
3,SE.PRM.NENR.FE,4940,63.17
4,school_enrollment,4067,52.01
5,pharma_imports,2999,38.35
6,oil_exports,2999,38.35
7,fuel_imports,2999,38.35
8,regime_score,2593,33.16
9,conflict_incidence,2593,33.16


The missingness table shows that several variables, especially `gini_index`, `poverty_rate`, and school enrollment indicators, have substantial missing values. This is common in international panel data and reflects uneven data availability across countries and years.

## Missing-Value Strategy

I do not apply one rule to every missing value. Instead, I use variable-specific logic.

For sanction-related variables, a missing value is reasonably interpreted as no active sanction in that country-year, so I fill those with default no-sanction values. For political, socioeconomic, and trade variables, I keep missing values as `NaN` at this stage because there is not yet a strong justification for replacing them with zeros or estimated values during basic cleaning.

In [9]:
# sanction-related fields:
# reasonable to fill with no-sanction defaults
clean_df["sanction_active"] = clean_df["sanction_active"].fillna(0)
clean_df["sanction_duration"] = clean_df["sanction_duration"].fillna(0)
clean_df["sanction_intensity_index"] = clean_df["sanction_intensity_index"].fillna(0)
clean_df["sanction_type"] = clean_df["sanction_type"].fillna("none")

# keep political variables as missing for now
political_cols = ["regime_score", "conflict_incidence", "conflict_intensity"]

print("Political missing values:")
print(clean_df[political_cols].isna().sum())

# keep socioeconomic and trade missing values for now
indicator_cols = [
    "inflation_rate", "gdp_growth", "school_enrollment",
    "SE.PRM.NENR.FE", "SE.PRM.NENR.MA",
    "child_mortality_u5", "SH.DYN.MORT.FE", "SH.DYN.MORT.MA",
    "poverty_rate", "gini_index", "unemployment_rate",
    "oil_exports", "pharma_imports", "fuel_imports"
]

print("\nIndicator missing values:")
print(clean_df[indicator_cols].isna().sum().sort_values(ascending=False))

Political missing values:
regime_score          2593
conflict_incidence    2593
conflict_intensity    2593
dtype: int64

Indicator missing values:
gini_index            5761
poverty_rate          5349
SE.PRM.NENR.FE        4940
SE.PRM.NENR.MA        4940
school_enrollment     4067
oil_exports           2999
pharma_imports        2999
fuel_imports          2999
inflation_rate        1080
unemployment_rate      779
child_mortality_u5     744
SH.DYN.MORT.FE         744
SH.DYN.MORT.MA         744
gdp_growth             232
dtype: int64


This step preserves an important distinction between missing values and actual zeros. In this dataset:

- `NaN` means the value is unavailable or the country-year observation did not have a matching source record
- `0` means the value is known and equal to zero, or was filled with a justified no-sanction default

This distinction is especially important for political variables, where replacing missing values with zero would incorrectly imply observed political conditions rather than unavailable data.

## Missing Values vs. Zero Values: Interpretation

Because this dataset was merged from several sources, both missing values and zeros appear in the final table. These should not be interpreted in the same way.

Missing values indicate that the relevant source dataset did not provide a value for that country-year. In contrast, zeros indicate either a genuine observed zero or a justified default value, especially for sanction-related variables where missingness is interpreted as no active sanction.

In [10]:
# values that should never be negative
non_negative_cols = [
    "school_enrollment", "SE.PRM.NENR.FE", "SE.PRM.NENR.MA",
    "child_mortality_u5", "SH.DYN.MORT.FE", "SH.DYN.MORT.MA",
    "poverty_rate", "gini_index", "unemployment_rate",
    "sanction_duration", "sanction_intensity_index",
    "oil_exports", "pharma_imports", "fuel_imports"
]

for col in non_negative_cols:
    clean_df.loc[clean_df[col] < 0, col] = np.nan

# values that should typically be between 0 and 100
bounded_0_100_cols = [
    "school_enrollment", "SE.PRM.NENR.FE", "SE.PRM.NENR.MA",
    "poverty_rate", "gini_index", "unemployment_rate"
]

for col in bounded_0_100_cols:
    clean_df.loc[(clean_df[col] < 0) | (clean_df[col] > 100), col] = np.nan

# sanction_active should be binary
clean_df.loc[~clean_df["sanction_active"].isin([0, 1, np.nan]), "sanction_active"] = np.nan

# conflict_incidence should be binary if present
clean_df.loc[~clean_df["conflict_incidence"].isin([0, 1, np.nan]), "conflict_incidence"] = np.nan

# conflict_intensity should not be negative
clean_df.loc[clean_df["conflict_intensity"] < 0, "conflict_intensity"] = np.nan

The examples above show that the presence of `NaN` values in political variables does not indicate a merge failure. Instead, it reflects differences in coverage across countries and years. For example, Afghanistan has political values available, while Aruba does not for the same period.

## Obvious Error Checks

I next check for values that are clearly invalid. Some variables should never be negative, such as mortality, trade amounts, and sanction duration. Other variables, such as enrollment, poverty, inequality, and unemployment, should generally lie within a 0 to 100 range when expressed as percentages or rates.

Invalid entries are replaced with missing values so that they do not silently distort later analysis.

In [11]:
print("Cleaned shape:", clean_df.shape)

print("\nDuplicate country-year rows after cleaning:")
print(clean_df[["country_code", "year"]].duplicated().sum())

print("\nMissing values after cleaning:")
print(clean_df.isna().sum().sort_values(ascending=False))

print("\nSample rows:")
clean_df.head()

Cleaned shape: (7820, 24)

Duplicate country-year rows after cleaning:
0

Missing values after cleaning:
gini_index                  5761
poverty_rate                5349
SE.PRM.NENR.MA              4940
SE.PRM.NENR.FE              4940
school_enrollment           4067
pharma_imports              2999
oil_exports                 2999
fuel_imports                2999
regime_score                2593
conflict_incidence          2593
conflict_intensity          2593
inflation_rate              1080
unemployment_rate            779
child_mortality_u5           744
SH.DYN.MORT.FE               744
SH.DYN.MORT.MA               744
gdp_growth                   232
country                        0
sanction_active                0
sanction_type                  0
sanction_duration              0
sanction_intensity_index       0
year                           0
country_code                   0
dtype: int64

Sample rows:


,country_code,country,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,...,sanction_active,sanction_type,sanction_duration,sanction_intensity_index,regime_score,conflict_incidence,conflict_intensity,oil_exports,pharma_imports,fuel_imports
0,ABW,ABW,1995,3.361391,2.547144,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,ABW,1996,3.225288,1.185789,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,ABW,1997,2.999948,7.046875,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,ABW,1998,1.869489,1.991984,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,ABW,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


This step does not necessarily change many values, but it is an important part of a trustworthy pipeline. It ensures that impossible values are explicitly flagged rather than silently retained.

## Missing Values and Zero Values: Interpretation

This dataset is a country-year panel built by using the WDI table as the base dataset and then merging sanctions, political, and trade data on country_code and year.

As a result, some columns contain missing values (NaN). These missing values do not automatically indicate an error. In most cases, they mean that a given country-year observation exists in the base dataset but does not have a matching record in one of the merged source datasets. For example, some countries appear in WDI but do not have political data for all years, so the political variables remain missing for those rows.

Zero values and missing values are interpreted differently:
- NaN means the value is unavailable or there was no matching observation in the source dataset.
- 0 means the value is known and equal to zero.

For sanction-related variables, missing values were filled with defensible defaults:
- sanction_active = 0
- sanction_duration = 0
- sanction_intensity_index = 0
- sanction_type = "none"

This choice was made because the absence of a sanction record is reasonably interpreted as no active sanction in that country-year.

Political and socioeconomic variables were not filled with zero at this stage, because doing so would incorrectly treat missing data as observed zero values. These variables were left as NaN unless there was a strong justification for a later imputation step.

In [12]:
print("Rows with available political data:", clean_df["regime_score"].notna().sum())
print("Rows with missing political data:", clean_df["regime_score"].isna().sum())

print("\nSanction value counts:")
print(clean_df["sanction_active"].value_counts(dropna=False))

print("\nExample rows with political data present:")
display(
    clean_df[clean_df["regime_score"].notna()][
        ["country_code", "year", "regime_score", "conflict_incidence", "conflict_intensity"]
    ].head(10)
)

print("\nExample rows with political data missing:")
display(
    clean_df[clean_df["regime_score"].isna()][
        ["country_code", "year", "regime_score", "conflict_incidence", "conflict_intensity"]
    ].head(10)
)

Rows with available political data: 5227
Rows with missing political data: 2593

Sanction value counts:
sanction_active
0.0    7507
1.0     313
Name: count, dtype: int64

Example rows with political data present:


,country_code,year,regime_score,conflict_incidence,conflict_intensity
60,AFG,1995,0.094,1.0,2.0
61,AFG,1996,0.076,1.0,2.0
62,AFG,1997,0.073,1.0,2.0
63,AFG,1998,0.073,1.0,2.0
64,AFG,1999,0.073,1.0,2.0
65,AFG,2000,0.073,1.0,2.0
66,AFG,2001,0.084,1.0,2.0
67,AFG,2002,0.220,1.0,1.0
68,AFG,2003,0.227,1.0,1.0
69,AFG,2004,0.238,1.0,1.0



Example rows with political data missing:


,country_code,year,regime_score,conflict_incidence,conflict_intensity
0,ABW,1995,NaN,NaN,NaN
1,ABW,1996,NaN,NaN,NaN
2,ABW,1997,NaN,NaN,NaN
3,ABW,1998,NaN,NaN,NaN
4,ABW,1999,NaN,NaN,NaN
5,ABW,2000,NaN,NaN,NaN
6,ABW,2001,NaN,NaN,NaN
7,ABW,2002,NaN,NaN,NaN
8,ABW,2003,NaN,NaN,NaN
9,ABW,2004,NaN,NaN,NaN


## Validation After Cleaning

After applying the cleaning rules, I validate the dataset again. The main checks are:
- whether the overall shape remained stable
- whether duplicate country-year rows are still absent
- whether the missing-value profile is consistent with the documented cleaning decisions

In [13]:
print("Cleaned shape:", clean_df.shape)

print("\nDuplicate country-year rows after cleaning:")
print(clean_df[["country_code", "year"]].duplicated().sum())

print("\nMissing values after cleaning:")
print(clean_df.isna().sum().sort_values(ascending=False))

print("\nSample rows:")
clean_df.head()

Cleaned shape: (7820, 24)

Duplicate country-year rows after cleaning:
0

Missing values after cleaning:
gini_index                  5761
poverty_rate                5349
SE.PRM.NENR.MA              4940
SE.PRM.NENR.FE              4940
school_enrollment           4067
pharma_imports              2999
oil_exports                 2999
fuel_imports                2999
regime_score                2593
conflict_incidence          2593
conflict_intensity          2593
inflation_rate              1080
unemployment_rate            779
child_mortality_u5           744
SH.DYN.MORT.FE               744
SH.DYN.MORT.MA               744
gdp_growth                   232
country                        0
sanction_active                0
sanction_type                  0
sanction_duration              0
sanction_intensity_index       0
year                           0
country_code                   0
dtype: int64

Sample rows:


,country_code,country,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,...,sanction_active,sanction_type,sanction_duration,sanction_intensity_index,regime_score,conflict_incidence,conflict_intensity,oil_exports,pharma_imports,fuel_imports
0,ABW,ABW,1995,3.361391,2.547144,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,ABW,1996,3.225288,1.185789,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,ABW,1997,2.999948,7.046875,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,ABW,1998,1.869489,1.991984,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,ABW,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


The validation step confirms that the dataset remains stable after cleaning. The country-year panel structure is preserved, no duplicate rows were introduced, and the remaining missing values are consistent with the documented decision to preserve unavailable political, socioeconomic, and trade values as missing.

## Before-vs-After Comparison

To summarize the effect of the cleaning pipeline, I compare missing-value counts before and after cleaning. In this case, the main changes are structural and interpretive rather than aggressive imputation, so large reductions in missingness are not expected.

In [14]:
before_missing = df.isna().sum().sort_values(ascending=False)
after_missing = clean_df.isna().sum().sort_values(ascending=False)

comparison = pd.DataFrame({
    "before_missing": before_missing,
    "after_missing": after_missing
}).fillna(0)

comparison

,before_missing,after_missing
gini_index,5761,5761
poverty_rate,5349,5349
SE.PRM.NENR.MA,4940,4940
SE.PRM.NENR.FE,4940,4940
school_enrollment,4067,4067
pharma_imports,2999,2999
oil_exports,2999,2999
fuel_imports,2999,2999
regime_score,2593,2593
conflict_incidence,2593,2593


The before-versus-after comparison shows that the cleaning stage mainly improved consistency, validation, and interpretability rather than artificially reducing missingness. This is appropriate because many missing values reflect genuine gaps in source coverage rather than simple formatting errors.

## Cleaning Report

This cleaning pipeline focused on making the merged country-year dataset consistent, interpretable, and ready for later modeling. The structure of the dataset was already strong after merging, so the goal here was to validate it and apply defensible cleaning rules rather than rebuild it.

First, I converted columns into appropriate types so that years, numeric indicators, and text-based variables behave correctly in analysis. I also standardized string formatting for `country_code`, `country`, and `sanction_type` to avoid hidden inconsistencies.

Next, I checked for duplicate rows using `country_code` and `year` as the uniqueness rule. No duplicate observations were found, confirming that the dataset preserves a valid panel structure.

I then reviewed missing values and applied variable-specific rules. Missing sanction fields were replaced with defensible defaults because absent sanction records are reasonably interpreted as no active sanction. Political, trade, and socioeconomic variables were left as missing when no defensible rule supported filling them during the basic cleaning stage.

Finally, I performed range checks for clearly invalid values, such as negative values in non-negative indicators and percentages outside the expected 0–100 range. Invalid entries were converted to missing values so they would be clearly flagged instead of silently retained.

## Save Analysis-Ready Dataset

After validation, I save the cleaned dataset as an analysis-ready file. This file will serve as the input for the next stage of the project, where I will engineer features for modeling.

In [15]:
clean_df.to_csv("../data/processed/analysis_ready_dataset.csv", index=False)

print("Saved: ../data/processed/analysis_ready_dataset.csv")
print("Final cleaned shape:", clean_df.shape)

Saved: ../data/processed/analysis_ready_dataset.csv
Final cleaned shape: (7820, 24)


In [16]:
analysis_df = pd.read_csv("../data/processed/analysis_ready_dataset.csv")

print("Reloaded shape:", analysis_df.shape)
analysis_df.head()

Reloaded shape: (7820, 24)


,country_code,country,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,...,sanction_active,sanction_type,sanction_duration,sanction_intensity_index,regime_score,conflict_incidence,conflict_intensity,oil_exports,pharma_imports,fuel_imports
0,ABW,ABW,1995,3.361391,2.547144,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,ABW,1996,3.225288,1.185789,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,ABW,1997,2.999948,7.046875,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,ABW,1998,1.869489,1.991984,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,ABW,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Conclusion

The cleaning stage produced a stable analysis-ready dataset with 7,820 country-year observations and 24 variables. The panel structure was preserved, duplicate rows were not present, text and numeric formats were standardized, and missing values were handled using variable-specific rules.

This cleaned dataset is now ready for Module 7, where the next step is feature engineering and creation of a model-ready table.

# Data Wrangling Part II: Feature Engineering

## Objective
The goal of this section is to transform the cleaned country-year dataset into a model-ready table for analyzing the relationship between sanctions and civilian well-being.

Because the project focuses on the economic and social effects of sanctions, the feature engineering choices below are designed to capture:
- sanction presence and severity
- trade exposure
- political and conflict context
- gender-related differences in education and mortality
- delayed effects over time

This stage does not create features just for the sake of adding variables. It creates features that are relevant to the project question and can be justified in later modeling.

In [17]:
feature_df = pd.read_csv("../data/processed/analysis_ready_dataset.csv")

print("Loaded analysis-ready dataset shape:", feature_df.shape)
feature_df.head()

Loaded analysis-ready dataset shape: (7820, 24)


,country_code,country,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,...,sanction_active,sanction_type,sanction_duration,sanction_intensity_index,regime_score,conflict_incidence,conflict_intensity,oil_exports,pharma_imports,fuel_imports
0,ABW,ABW,1995,3.361391,2.547144,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,ABW,1996,3.225288,1.185789,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,ABW,1997,2.999948,7.046875,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,ABW,1998,1.869489,1.991984,NaN,NaN,NaN,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,ABW,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,...,0.0,none,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Sort the Panel Data

Since this dataset is a country-year panel, I sort it by `country_code` and `year` before creating lagged or change-based features. This ensures that time-based calculations are done in the correct order within each country.

In [18]:
feature_df = feature_df.sort_values(["country_code", "year"]).reset_index(drop=True)

feature_df[["country_code", "year"]].head(10)

,country_code,year
0,ABW,1995
1,ABW,1996
2,ABW,1997
3,ABW,1998
4,ABW,1999
5,ABW,2000
6,ABW,2001
7,ABW,2002
8,ABW,2003
9,ABW,2004


## Project-Relevant Missing-Value Handling

To create useful engineered features, some additional missing-value handling is needed. I still avoid blanket imputation, but I fill values in a way that is more defensible for a country-year panel.

For political variables, I use forward-fill and backward-fill within each country so nearby years provide context when values are missing. For continuous socioeconomic and trade indicators, I interpolate within each country over time so that derived features such as gaps, lags, and changes can be computed more consistently.

In [19]:
# political variables: fill within country using nearby years
political_cols = ["regime_score", "conflict_incidence", "conflict_intensity"]

feature_df[political_cols] = (
    feature_df.groupby("country_code")[political_cols]
    .transform(lambda x: x.ffill().bfill())
)

# continuous indicators: interpolate within each country across time
continuous_cols = [
    "inflation_rate", "gdp_growth", "school_enrollment",
    "SE.PRM.NENR.FE", "SE.PRM.NENR.MA",
    "child_mortality_u5", "SH.DYN.MORT.FE", "SH.DYN.MORT.MA",
    "poverty_rate", "gini_index", "unemployment_rate",
    "oil_exports", "pharma_imports", "fuel_imports"
]

feature_df[continuous_cols] = (
    feature_df.groupby("country_code")[continuous_cols]
    .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

print(feature_df.isna().sum().sort_values(ascending=False).head(15))

gini_index            2721
conflict_intensity    2570
conflict_incidence    2570
regime_score          2570
poverty_rate          2271
pharma_imports        1975
oil_exports           1975
fuel_imports          1975
SE.PRM.NENR.MA         901
SE.PRM.NENR.FE         901
unemployment_rate      770
school_enrollment      690
inflation_rate         621
child_mortality_u5     502
SH.DYN.MORT.FE         502
dtype: int64


## Create Derived Features for Civilian Well-Being

The project focuses on how sanctions relate to civilian well-being, so I create features that reflect economic and social stress more directly.

These derived features capture:
- gender gaps in school enrollment
- gender gap in child mortality
- combined trade exposure
- interactions between sanctions and political/conflict conditions

These variables are designed to make the later model more informative and better aligned with the project question.

In [20]:
# gender gap in school enrollment
feature_df["school_enrollment_gap"] = (
    feature_df["SE.PRM.NENR.FE"] - feature_df["SE.PRM.NENR.MA"]
)

# gender gap in child mortality
feature_df["child_mortality_gap"] = (
    feature_df["SH.DYN.MORT.MA"] - feature_df["SH.DYN.MORT.FE"]
)

# total trade exposure
feature_df["total_trade_exposure"] = (
    feature_df["oil_exports"] +
    feature_df["pharma_imports"] +
    feature_df["fuel_imports"]
)

# sanctions interacting with political regime
feature_df["sanction_x_regime"] = (
    feature_df["sanction_intensity_index"] * feature_df["regime_score"]
)

# sanctions interacting with conflict
feature_df["sanction_x_conflict"] = (
    feature_df["sanction_intensity_index"] * feature_df["conflict_intensity"]
)

feature_df[[
    "school_enrollment_gap",
    "child_mortality_gap",
    "total_trade_exposure",
    "sanction_x_regime",
    "sanction_x_conflict"
]].head()

,school_enrollment_gap,child_mortality_gap,total_trade_exposure,sanction_x_regime,sanction_x_conflict
0,1.27366,NaN,27909151.0,NaN,NaN
1,1.27366,NaN,27909151.0,NaN,NaN
2,1.27366,NaN,27909151.0,NaN,NaN
3,1.27366,NaN,27909151.0,NaN,NaN
4,1.27366,NaN,27909151.0,NaN,NaN


## Create Lag Features

Sanctions often affect countries with a delay rather than only in the same year. To reflect that, I create one-year lag features for several key economic, social, and sanction-related variables.

These lagged variables help the later model capture how prior conditions may influence current civilian well-being outcomes.

In [21]:
lag_cols = [
    "inflation_rate",
    "gdp_growth",
    "poverty_rate",
    "unemployment_rate",
    "child_mortality_u5",
    "sanction_intensity_index"
]

for col in lag_cols:
    feature_df[f"{col}_lag1"] = feature_df.groupby("country_code")[col].shift(1)

feature_df[[f"{col}_lag1" for col in lag_cols]].head(10)

,inflation_rate_lag1,gdp_growth_lag1,poverty_rate_lag1,unemployment_rate_lag1,child_mortality_u5_lag1,sanction_intensity_index_lag1
0,NaN,NaN,NaN,NaN,NaN,NaN
1,3.361391,2.547144,NaN,NaN,NaN,0.0
2,3.225288,1.185789,NaN,NaN,NaN,0.0
3,2.999948,7.046875,NaN,NaN,NaN,0.0
4,1.869489,1.991984,NaN,NaN,NaN,0.0
5,2.280372,1.238042,NaN,NaN,NaN,0.0
6,4.044021,7.622921,NaN,NaN,NaN,0.0
7,2.883604,4.182002,NaN,NaN,NaN,0.0
8,3.315247,-0.944953,NaN,NaN,NaN,0.0
9,3.656365,1.110505,NaN,NaN,NaN,0.0


The lag features behave as expected in a country-year panel. The first observation for each country has no previous year, so its lag values are missing. Additional missing lag values appear when the original variable was already missing in the prior year. In contrast, lagged sanction intensity often takes the value 0 because many country-years had no active sanction in the previous year.

## Create Change Features

In addition to lagged levels, I create year-to-year change features for selected economic indicators. These variables measure how conditions are changing over time within each country, which is useful because sanctions may be associated with worsening or improving trends rather than only static levels.

In [22]:
change_cols = [
    "gdp_growth",
    "inflation_rate",
    "poverty_rate",
    "unemployment_rate"
]

for col in change_cols:
    feature_df[f"{col}_change"] = feature_df.groupby("country_code")[col].diff()

feature_df[[f"{col}_change" for col in change_cols]].head(10)

,gdp_growth_change,inflation_rate_change,poverty_rate_change,unemployment_rate_change
0,NaN,NaN,NaN,NaN
1,-1.361355,-0.136103,NaN,NaN
2,5.861086,-0.225340,NaN,NaN
3,-5.054891,-1.130460,NaN,NaN
4,-0.753943,0.410883,NaN,NaN
5,6.384879,1.763649,NaN,NaN
6,-3.440919,-1.160417,NaN,NaN
7,-5.126955,0.431642,NaN,NaN
8,2.055458,0.341118,NaN,NaN
9,6.183223,-1.127236,NaN,NaN


## Transform Skewed Trade Features

Trade variables often have very large ranges across countries, which can make them highly skewed. To reduce the effect of extreme values while preserving the ordering of observations, I apply a log transform using `log1p`.

This transformation is especially useful for trade-related variables such as oil exports, pharmaceutical imports, fuel imports, and total trade exposure.

In [23]:
for col in ["oil_exports", "pharma_imports", "fuel_imports", "total_trade_exposure"]:
    feature_df[f"log_{col}"] = np.log1p(feature_df[col])

feature_df[[
    "log_oil_exports",
    "log_pharma_imports",
    "log_fuel_imports",
    "log_total_trade_exposure"
]].head()

,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure
0,0.0,17.144465,0.0,17.144465
1,0.0,17.144465,0.0,17.144465
2,0.0,17.144465,0.0,17.144465
3,0.0,17.144465,0.0,17.144465
4,0.0,17.144465,0.0,17.144465


## Encode Categorical Variables

The main categorical variable relevant to this project is `sanction_type`. Since it is a nominal category rather than an ordered one, one-hot encoding is more appropriate than label encoding.

I also create a simple `sanction_status` variable to distinguish sanctioned and non-sanctioned country-years more clearly.

In [24]:
feature_df["sanction_status"] = np.where(
    feature_df["sanction_active"] == 1,
    "sanctioned",
    "not_sanctioned"
)

feature_df = pd.get_dummies(
    feature_df,
    columns=["sanction_type", "sanction_status"],
    drop_first=True
)

print("Shape after encoding:", feature_df.shape)
feature_df.head()

Shape after encoding: (7820, 46)


,country_code,country,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,...,poverty_rate_change,unemployment_rate_change,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure,sanction_type_none,sanction_type_targeted,sanction_type_trade,sanction_status_sanctioned
0,ABW,ABW,1995,3.361391,2.547144,97.82877,98.47424,97.20058,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
1,ABW,ABW,1996,3.225288,1.185789,97.82877,98.47424,97.20058,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
2,ABW,ABW,1997,2.999948,7.046875,97.82877,98.47424,97.20058,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
3,ABW,ABW,1998,1.869489,1.991984,97.82877,98.47424,97.20058,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
4,ABW,ABW,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False


## Feature Selection for Modeling

At this point, I create a model-ready table by removing identifier columns that are useful for grouping and interpretation but should not be used as predictive inputs.

The columns `country_code` and `country` function as identifiers rather than meaningful explanatory variables, so I remove them from the model-ready dataset.

In [25]:
model_ready_df = feature_df.copy()

drop_cols = ["country_code", "country"]

model_ready_df = model_ready_df.drop(columns=drop_cols, errors="ignore")

print("Model-ready dataset shape:", model_ready_df.shape)
model_ready_df.head()

Model-ready dataset shape: (7820, 44)


,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,SH.DYN.MORT.MA,poverty_rate,...,poverty_rate_change,unemployment_rate_change,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure,sanction_type_none,sanction_type_targeted,sanction_type_trade,sanction_status_sanctioned
0,1995,3.361391,2.547144,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
1,1996,3.225288,1.185789,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
2,1997,2.999948,7.046875,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
3,1998,1.869489,1.991984,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
4,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False


## Optional Standardization

Many machine learning models perform better when numeric features are on comparable scales. In this step, I standardize the numeric predictors to create a scaled version of the model-ready dataset.

I keep `year` unscaled so that it remains interpretable as a time variable.

In [26]:
from sklearn.preprocessing import StandardScaler

scaled_df = model_ready_df.copy()

numeric_features = scaled_df.select_dtypes(include=[np.number]).columns.tolist()

if "year" in numeric_features:
    numeric_features.remove("year")

scaler = StandardScaler()
scaled_df[numeric_features] = scaler.fit_transform(scaled_df[numeric_features])

scaled_df.head()

,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,SH.DYN.MORT.MA,poverty_rate,...,poverty_rate_change,unemployment_rate_change,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure,sanction_type_none,sanction_type_targeted,sanction_type_trade,sanction_status_sanctioned
0,1995,-0.088198,-0.166753,0.796036,0.87696,0.801143,NaN,NaN,NaN,NaN,...,NaN,NaN,-2.494052,-0.437102,-5.610308,-1.366785,True,False,False,False
1,1996,-0.090310,-0.398369,0.796036,0.87696,0.801143,NaN,NaN,NaN,NaN,...,NaN,NaN,-2.494052,-0.437102,-5.610308,-1.366785,True,False,False,False
2,1997,-0.093806,0.598816,0.796036,0.87696,0.801143,NaN,NaN,NaN,NaN,...,NaN,NaN,-2.494052,-0.437102,-5.610308,-1.366785,True,False,False,False
3,1998,-0.111346,-0.261206,0.796036,0.87696,0.801143,NaN,NaN,NaN,NaN,...,NaN,NaN,-2.494052,-0.437102,-5.610308,-1.366785,True,False,False,False
4,1999,-0.104971,-0.389479,0.796036,0.87696,0.801143,NaN,NaN,NaN,NaN,...,NaN,NaN,-2.494052,-0.437102,-5.610308,-1.366785,True,False,False,False


## Validation of the Feature Table

Before saving the feature-engineered output, I validate the result by checking:
- the final shape of each dataset
- the remaining missing values
- whether the model-ready versions were created successfully

This step makes sure the next notebook can load the table directly for modeling.

In [27]:
print("Feature-engineered shape:", feature_df.shape)
print("Model-ready shape:", model_ready_df.shape)
print("Scaled model-ready shape:", scaled_df.shape)

print("\nTop remaining missing values in model-ready dataset:")
print(model_ready_df.isna().sum().sort_values(ascending=False).head(15))

Feature-engineered shape: (7820, 46)
Model-ready shape: (7820, 44)
Scaled model-ready shape: (7820, 44)

Top remaining missing values in model-ready dataset:
gini_index                  2721
conflict_incidence          2570
sanction_x_conflict         2570
sanction_x_regime           2570
conflict_intensity          2570
regime_score                2570
poverty_rate_change         2456
poverty_rate_lag1           2456
poverty_rate                2271
pharma_imports              1975
log_pharma_imports          1975
log_oil_exports             1975
total_trade_exposure        1975
fuel_imports                1975
log_total_trade_exposure    1975
dtype: int64


## Before-vs-After Feature Summary

To document the effect of feature engineering, I compare the size of the cleaned dataset with the size of the feature-engineered and model-ready datasets.

This helps show how many new variables were added and confirms that the project now has a dedicated table ready for modeling.

In [28]:
print("Analysis-ready shape:", pd.read_csv("../data/processed/analysis_ready_dataset.csv").shape)
print("Feature-engineered shape:", feature_df.shape)
print("Model-ready shape:", model_ready_df.shape)
print("Scaled model-ready shape:", scaled_df.shape)

Analysis-ready shape: (7820, 24)
Feature-engineered shape: (7820, 46)
Model-ready shape: (7820, 44)
Scaled model-ready shape: (7820, 44)


## Feature Engineering Report

This stage transformed the cleaned country-year dataset into a model-ready feature table aligned with the project objective of analyzing the economic impact of international sanctions on civilian well-being.

I created derived features that capture gender gaps, trade exposure, and interactions between sanctions and political conditions. I also created lagged and change-based features to reflect the idea that sanctions may affect countries with a delay rather than only in the same year.

Categorical sanction information was one-hot encoded so that models can use it correctly. Trade variables were log-transformed to reduce skewness, and identifier columns were removed from the final model-ready table because they are not appropriate predictive inputs.

The final output preserves the country-year structure while providing a richer set of features for the modeling stage.

## Save the Feature-Engineered Outputs

I save three outputs from this stage:
- `feature_engineered_dataset.csv` as the enriched dataset with engineered variables
- `model_ready_dataset.csv` as the final modeling table
- `model_ready_scaled_dataset.csv` as the standardized version for models that benefit from scaling

In [29]:
feature_df.to_csv("../data/processed/feature_engineered_dataset.csv", index=False)
model_ready_df.to_csv("../data/processed/model_ready_dataset.csv", index=False)
scaled_df.to_csv("../data/processed/model_ready_scaled_dataset.csv", index=False)

print("Saved: ../data/processed/feature_engineered_dataset.csv")
print("Saved: ../data/processed/model_ready_dataset.csv")
print("Saved: ../data/processed/model_ready_scaled_dataset.csv")

Saved: ../data/processed/feature_engineered_dataset.csv
Saved: ../data/processed/model_ready_dataset.csv
Saved: ../data/processed/model_ready_scaled_dataset.csv


## Reload Check

As a final verification step, I reload the saved model-ready dataset to confirm that it was written correctly and can be used directly in the modeling notebook.

In [30]:
final_model_df = pd.read_csv("../data/processed/model_ready_dataset.csv")

print("Reloaded model-ready shape:", final_model_df.shape)
final_model_df.head()

Reloaded model-ready shape: (7820, 44)


,year,inflation_rate,gdp_growth,school_enrollment,SE.PRM.NENR.FE,SE.PRM.NENR.MA,child_mortality_u5,SH.DYN.MORT.FE,SH.DYN.MORT.MA,poverty_rate,...,poverty_rate_change,unemployment_rate_change,log_oil_exports,log_pharma_imports,log_fuel_imports,log_total_trade_exposure,sanction_type_none,sanction_type_targeted,sanction_type_trade,sanction_status_sanctioned
0,1995,3.361391,2.547144,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
1,1996,3.225288,1.185789,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
2,1997,2.999948,7.046875,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
3,1998,1.869489,1.991984,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False
4,1999,2.280372,1.238042,97.82877,98.47424,97.20058,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,17.144465,0.0,17.144465,True,False,False,False


## Conclusion

The feature engineering stage created a model-ready dataset aligned with the project question. The new features capture sanction severity, trade exposure, political and conflict context, gender-related gaps, and time-based effects through lag and change variables.

This output is now ready to be used in `data_modeling.ipynb`.